## Collecting Data from the Web

In [ ]:
import pandas as pd

When dealing with text in data/code we often want to represent special characters, which might have a weird effect in our code
* For example the double quote " or single quote might be something we want to incorporate into our string, but not want to write
* Similarly with carriage returns/newlines

A lot of this is built into many programming/coding languages, including Python

For example think about writing the following HTML string in Python:

In [ ]:
stringIn="<h2>Hello</h2>\n\n\t<div id=\"main\">Main text inside a content divider</div>\n\n<footer style=\"color:#003594;background-color:#FFB81C\"><center>Bottom footer text in Blue</center></footer>\\"

When this is printed or written to a data file though, this same string would look like this:

In [ ]:
print(stringIn)

Here I'm using the code: 
* `\n` in place of a newline
* `\t` in place of a Tab
* `\"` in pace of a double quote

Because `\` is used here as an escape character if we want to use the **actual** `\` character in a string we need to type `\\`

As an aside, whenever you use strings in Python you do need to be careful with certain characters, which is why we were compiling the expressions above first.

Because the backslash character is special within the "regular expressions" we will talk about below, we need to make sure that when we pass them to the `re` module that it's seeing the same thing as us
* Anytime "\\" appears in the string, it is telling the compiler that this is something special. 
* If you're actually looking for "\\" then you need to escape it as "\\\\" within a Python string,  as Python will interpret `\\` the special backslash character on its own; telling the regular expression to look for the special string  "\\"
* Some useful escape characters are: 
    - `\'` is '
    - `\"` is " 
    
*Note: if you look at the Markdown for this cell, I wasn't able to write these strings directly unless they were in the code quotes!*

The same raw strings can be interpreted in HTML. Here I'm doing this in the below cell by using a *magic*, something which tells IPython that renders this workbook to not run the cell in python,  but to instead render this cell as HTML.

HTML is it's another markup language which has its own special characters, here anythings inside a tag: `<  >`

In [ ]:
%%HTML 
<h2>Hello</h2>

	<div id="main">Main text inside a content divider</div>

<footer style="color:#003594;background-color:#FFB81C"><center>Bottom footer text in Blue</center></footer>

While we can write pure HTML and style it, websites will often use something called a cascading style sheet (CSS). This is a list of rules about how to format HTML objects with certain properties. Having a systematic styling scheme often means that websites will give elements of the HTML special identifiers like the `id` in the `<div>` tag above. This structure can be useful when we want to extract data.

However, when we call the string in Python (without printing it) it is simply the characters we entered:

In [ ]:
stringIn

## Search Patterns

In addition to knowing how to enter these special characters, escape characters are also used in flexible search patterns called **regular expressions**. These highly stuctured patterns can then be used to look for and extract data.

**Proviso:** I am not an expert at using regular expressions. I know they exist, and I know how to use them *a little*. Almost all the time I have a special use for a complex regular expression, a little googling finds me the answer. 

While I can do basic regular expressions on the go, when I'm doing something more complicated I mostly go to [this website](https://regexr.com/) and paste in a sample of the the data I'm using, to check/build the regular expression!
* That way I can paste in some sample data and see what the regular expression will pick up!
* These regular expression are supplemented with additional structure that comes from a language called PERL, which allows for even finer detail.

We'll use the following string as a search pattern:

In [ ]:
searchString="My office number used to be (412) 383 8152, if you wanted to call my office, but you now can only send an email to instructor@pitt.edu to contact me.\n\n The main number for the MQE is +1-412-383-5425. I think Randy's office number is 412.648.1737. \n\nThe department has a twitter account at @PittEcon (look for the #PittMQE tag). Sometimes email addresses are like George's gl20@andrew.cmu.edu.\n"
print(searchString)

And we'll check what they return by using the `re` library in Python:

In [ ]:
# import regular expressions library
import re # this is actually a library from C

## Some examples:

* `(?:\s)?([\w\.-]+@[\w\.-]+\.\w{2,4})(?:\s\.)?`
    - `(?: )` :  represents a group we don't want, but is necessary to the search
    - `(  )` : without the question mark means a group to return in the output
    - `\s` means a space
    - After any character or group:
        - `?` after a character or group means 0 or 1 occurences
        - `+` means 1 or more occurences
        - `*` means 0 or more.
    - `[\w\.- ]+` : means  word characters, dots (plain dots, so escaped) or a dash,  repeated one or more times (the `+` modifier)
    - `@` has no special meaning here, it just means we require the "@" character in the pattern
    - `\w{2,4}` : means a word character, appearing two to four times (so "com", "gov", "ca").

So this will hopefully match email addresses.

In [ ]:
print(searchString)
emailFinder=re.compile("(?:\s)?([\w\.-]+@[\w\.-]+\.\w{2,4})(?:\s\.)?", re.S )
re.findall( emailFinder,searchString )

Twitter:
* `(@[A-z]+)|(#[A-z]+)`
    - `@` : The @ character
    - `[A-z]+` any number of characters A-Z or a-z
    - `|` : Boolean OR
    - `#` : the hash symbol

So this will match twitter user names (the first part) **or** hashtags (the second part)

In [ ]:
print(searchString)
tweetFinder=re.compile('(?:\s)(@[A-z]+)|(#[A-z]+)', re.S )
re.findall( tweetFinder,searchString )

What would we need to do to get rid of the entries we don't want?

You can also extract separated groups:
* `(?:\d{1}\s)?\(?(\d{3})\)?-?\s?.?(\d{3})-?\s?.?(\d{4})`
    - `(?: )` this allows this to be there, but doesn't retain it in the capture (so a country code or a space)
    - `?` when outside the parentheses this means either 0 or 1 repetition of the prior block
    - `\(?` : so either an open parenthesis or not
    - `(  )`: when the parentheses aren't escaped with `\` or put in a non captured group `(?  )`, this is saying we want to get this in the output
    - `\d{3}` : exactly three digits 0-9
    - `\)?` : 0 or 1 closing parenthesis
    - `-?`  , `\s?`: possibly a dash, possibly a space
    - `(\d{3})` : three more digits, capture them
    - `(\d{4}))` : four digits capture them

In [ ]:
print(searchString)
phoneFinder=re.compile("(?:\d{1}\s)?\(?(\d{3})\)?-?\s?.?(\d{3})-??\s?.?(\d{4})")
re.findall(phoneFinder,searchString)

Why did we not find the last number? How can we fix it?

In [ ]:
searchString

## Looking at HTML

Consider the HTML code:
```
    <table>
         <tr><td type="normal"><b>Column 1</b></td><td><b>Column 2</b></td> </tr>
        <tr><td type="normal">Entry 1</td><td>Entry 2</td> </tr>
        <tr><td type="other">Entry 3</td><td>Entry 4 </td></tr>
    </table>
```

It creates a simple structured table, and looks like this when rendered: (again using a magic, and without any fancy table styling commands)

In [ ]:
%%HTML
<table>
    <tr><td type="normal"><b>Column 1</b></td><td><b>Column 2</b></td> </tr>
    <tr><td type="normal">Entry 1</td><td>Entry 2</td> </tr>
    <tr><td type="other">Entry 3</td><td>Entry 4 </td></tr>
</table>

So if we wanted to try and get at some of the table data here:

* `(?:<td.*?>)(.*?)(?:<\/td>)`
    - So this asks us to match but not keep a `<td` string first followed by 
        - `.` means any character (as there may very well be a lot of styling information within the `<td>` tag 
        - `*` means we repeat the any character any number of times 
        - but `*?` means as few as possible, so we'll try and find the closing `>` tag as soon as we encounter one
    - `(.*?)` is our main data capture group, and so this will capture everything between the `<tr>` and the closing `</td>` tag (not `*?` specifies as few as possible again)
    - `(?:<\/td>)` : says don't capture the closing tag `</td>`, but that it's necessary

But when you apply our regular expression it will hopefully extract six entries:
* `<b>Column 1</b>`
* `<b>Column 2</b>`
* `Entry 1`
* `Entry 2`
* `Entry 3`
* `Entry 4`

In [ ]:
tableString='<table>\n\t<tr><td type="normal"><b>Column 1</b></td><td><b>Column 2</b></td> </tr>\n\t<tr><td type="normal">Entry 1</td><td>Entry 2</td> </tr>\n\t<tr><td type="other">Entry 3</td><td>Entry 4 </td></tr>\n</table>'
tdFinder=re.compile('(?:<td.*?>)(.*?)(?:<\/td>)')
pd.DataFrame({"found": re.findall(tdFinder,tableString)}) # I just add the list to a dict to label the column in the dataframe

Often times in web data, we can use the styling information to pick out specific types of data entries.

For example, suppose that we only wanted to extract the `normal` type `td` entries?

In [ ]:
tdFinder=re.compile('(?:<td.*?type=\"normal\".*?>)(.*?)(?:<\/td>)')
re.findall(tdFinder,tableString)

Regular expressions used to be the backbone of how you'd conduct a search in a website to extract information you wanted.

And they're still very useful to know about when for looking for patterns in text, particularly if you don't have more structure so let's go through how to extract web data with some regular expressions.

We can run regular expressions on web data within Python fairly easily, where we're already played with the `re` library a bit above. Let's load `requests` so we can fetch web pages.

In [ ]:
import requests # This loads the library that gives us the `get` method

Let's look at the course [home page](https://www.mqe.pitt.edu/courses) for the MQE. In terms of when you're building these expressions, I always find if useful to load the web page in a browser and open the variable inspector!

In [ ]:
response=requests.get('https://www.mqe.pitt.edu/economics-data-science-communication-courses')
response

In [ ]:
WebsiteHTML=response.text

Now we load the response as raw text to look through

In [ ]:
WebsiteHTML=response.text
print(WebsiteHTML[500:1000]) # show the first 3001 characters

When we load objects as regular expressions in python, we need to tell it to treat the string as a raw string by prefacing it with  `r`, that way we don't have to escape all the special characters we'd want to use or compile it separately.

In [ ]:
print('This is not a \n\traw string so the new lines aren\'t shown')
print(r'This is a \n\traw string so the new lines aren\'t shown')

It's possible to call a regex string directly without compiling it so long as it's a raw string. However, if you're going to use a regex many times you will be better off compiling it.

We'll use the `re` function to look through the data again, where we'll add three additional flags to the search:
* `re.M` : multi-line, so the pattern will reset when looking across multiple lines
* `re.I` : ignore case for characters, so A and a will be treated the same
* `re.S` : This flag means that the special character `.` that matches everything also includes newlines 

In [ ]:
matchTableData = re.findall(r'(?:<td.*?>)(.*?)(?:<\/td>)', WebsiteHTML, re.M|re.I|re.S)
print(len(matchTableData))
type(matchTableData)

In [ ]:
print(matchTableData[0])
print(matchTableData[1])

Define a function that strips our  anything matching a pattern search, where we'll use a search for matching tags to strip out all the html.

In [ ]:
tagMatch = re.compile('<.*?>', re.S) #
def stripText(text,match=tagMatch,replace=''): 
    # define a function to remove any HTML tags
    return re.sub(match, replace, text )

In [ ]:
print(tableString)

In [ ]:
print(stripText(tableString,tagMatch))

Or being slightly fancier:

In [ ]:
whitespaceMatch = re.compile('\s+', re.S) #
print(stripText(stripText(tableString,tagMatch," ") ,whitespaceMatch," "))

In [ ]:
matchTableDataList=[]
for item in matchTableData: # for each entry in our table search
    # Here I just extract hyperlink tags that look like <a href="#link-location">Label</a>. There point to internal links.
    tupleIn=re.findall(r'(?:<a.*?)href=\"#?(.*?)\"(?:.*?)>(.*?)(?:<\/a>)', item, re.M|re.I|re.S)
    for links in tupleIn: # now I look at each of the entries I found in this match
        # Here I construct a new regular expression using the relative link location I found
        # This looks for the H4 tags which contain the targets for the link
        # I then move into the <p> tag below it to get the course description.
        searchString=re.compile(r'(?:<h4.*?>.*?<a\s.*?id=\"'+links[0]+ r'\".*?>.*?<\/h4>.*?<p.*?>)(.*?)(?:<\/p>)',re.I|re.S)
        desc=re.findall( searchString, WebsiteHTML)
        # I then put this into a dict so that when I convert this to a dataframe it will already be set up correctly
        matchTableDataList.append({"location":links[0], "label": links[1],"description":stripText(desc[0])})
# Convert to dataframe
courseInformation=pd.DataFrame(matchTableDataList)

In [ ]:
courseInformation

Export to a csv

In [ ]:
courseInformation.to_csv('data/CousreInf.csv')

And so with only some knowledge for how the web page is constructed, I've extracted the course listing across multiple locations from the MQE website!

In [ ]:
print(courseInformation['label'][6])
print(courseInformation['description'][6])

## Other things to look for:

We can look for table rows...

In [ ]:
matchTableRow = re.findall(r'<tr.*?>(.*?)<\/tr>', WebsiteHTML, re.M|re.I|re.S)
matchTableRow[0]

And strip out some of the white space either side...

In [ ]:
matchTableRow[0].strip()

Or remove both tags and white space.

In [ ]:
stripText(stripText(matchTableRow[0] ,tagMatch," "), whitespaceMatch," ")

We can find the table headers

In [ ]:
matchTableHeader = re.findall(r'<th.*?>(.*?)<\/th>', WebsiteHTML, re.M|re.I|re.S)
matchTableHeader

In [ ]:
match_list=[]
for i in range(0,len(matchTableRow)):
    match_list.append( re.findall(r'<td.*?>.*<a.*>(.*?)<\/a>.*<\/td>', matchTableRow[i], re.M|re.I|re.S) )
pd.DataFrame(match_list)

We can also index where we found things...

In [ ]:
outTable=[]
matchTable =re.findall(r'<table.*?>(.*?)<\/table>', WebsiteHTML, re.M|re.I|re.S)
for i in range(0,len(matchTable)):
    matchTableRow =re.findall(r'<tr.*?>(.*?)<\/tr>',matchTable[i], re.M|re.I|re.S)
    for j in range(0,len(matchTableRow)):
        matchTableData=re.findall(r'<td.*?>(.*?)<\/td>',matchTableRow[j], re.M|re.I|re.S)
        for k in range(0,len(matchTableData)):
            outTable.append({"Table": i+1, "Row": j+1, "Cell" : k+1 , "Content" : stripText(matchTableData[k])  })
pd.DataFrame(outTable)

## More modern methods
Even though knowing regular expressions can be really useful, if we're dealing with HTML, there are other ways to get structured information out of it.

We'll briefly look at two libraries that can do this:
*  Beautiful Soup
* lxml.html

Though hopefully Andy taught you more about these!

In [ ]:
from bs4 import BeautifulSoup
# Convert out Website HTML into a
soup=BeautifulSoup(WebsiteHTML,'html.parser')
type(soup)

In [ ]:
soupImages=soup.findAll("img")
for image in soupImages:
    print(image["src"])

In [ ]:
soupLinks=soup.findAll("a", attrs = {'href' : True})
for link in soupLinks:
    print(link["href"])

In [ ]:
# Use a regular expression in the find for any href which has an internal link 
soupLinksInternal=soup.findAll("a", {"href": re.compile("^\/.*$") })
for link in soupLinksInternal:
    print(link["href"])

In [ ]:
# this just inverts the for syntax, to look at tll of the children within the soup object
for item in list(soup.children):
    print(type(item) )

In [ ]:
soupHTML=list(soup.children)[2]
for item in list(soupHTML.children):
    print(type(item) )
# list(soupHTML.children)[1] this looks like scripts at the tope
# list(soupHTML.children)[3] this looks like the main content.
mainBody=list(list(soupHTML.children)[3].children)
for item in list(soupHTML.children):
    print(type(item) )

So though we can continue to break the HTML code into parseable sections, we can also quickly filter out particular types of tags

In [ ]:
for row in soup.select('table td'):
    print(row)

In [ ]:
for row in soup.select('table td'):
    print(row.get_text())

In [ ]:
for row in soupHTML.select('table th'):
    print(row.get_text()) # Here using the get_text() method to remove the tags

In [ ]:
soup.find_all(id='global-econ')[0] #looking for any tag with a particular id

Here we select all of the `<a>` tags withing an `<h4>` tag

In [ ]:
soup.select("h4 a")

Another package is `lxml`

In [ ]:
%pip install lxml

In [ ]:
import lxml.html as lh
import pandas as pd
doc=lh.fromstring(WebsiteHTML)

In [ ]:
listAll=[]
tr_elems=doc.xpath('//tr') # look for tr tags
for i in range(0,len(tr_elems)): # Loop across the table row elements
    dictI={} # create an empty dictionary
    for j  in range( 0 , len( tr_elems[i] ) ): # for each of the elements in the row j =0 to n-1
        dictI[j+1]= tr_elems[i][j].text_content() # Add the entry to key j 
    listAll.append( dictI  )
pd.DataFrame(listAll)

Let's take a look at grabbing some Hockey information from the table on [Cap Friendly](https://www.capfriendly.com/) front page

In [ ]:
response=requests.get("https://www.capwages.com/")

In [ ]:
response.status_code

In [ ]:
capFriendlyHTML=response.text
capFriendlyHTML[0:200]

In [ ]:
cfDoc=lh.fromstring(capFriendlyHTML) #using lxml.html
cfsoup=BeautifulSoup(capFriendlyHTML,'html.parser') #using Beautiful Soup

Looking at the website, this looks like there's one big table, and several small ones, the fourth one has all the cap information.

In [ ]:
table_soup[0]

In [ ]:
table_soup=cfsoup.select('table')
for child in list(table_soup[2].children):
    print(child)
    

In [ ]:
listAll=[]
table_elems=cfDoc.xpath('//table')
tr_elems=cfDoc.xpath('//tr')
table_elems[1].text_content()

In [ ]:
listAll

In [ ]:
for i in range(0,len(tr_elems)):
    if len(tr_elems[i])==8: # let's only select table rows with eight elements (which should be the team cap hits by year)
        dictI={} # create an empty dictionary
        for j  in range( 0 , len( tr_elems[i] ) ): # for each of the elements in the row j =0 to n-1
            dictI[j+1]= tr_elems[i][j].text_content() # Add the entry to key j 
        listAll.append( dictI  )  # add the dictionary to the list
mainTable=pd.DataFrame(listAll[1:]) # get rid of row zero from the data
colNames= [str.lower() for str in listAll[0].values()] 

mainTable.columns=colNames # reasssign column names

Let's grab the team names

In [ ]:
teams=mainTable["team"].unique()
teams

Note that we can do for loops inside of list of dict definitions in python so that:

In [ ]:
my_list=[str.lower() for str in listAll[0].values() ] 
my_list

Is equivalent to doing it one by one:

In [ ]:
my_list=[]
for str in listAll[0].values():
    my_list.append (str.lower())
my_list

In [ ]:
mainTable

## BBC Sports Tables

In [ ]:
resp=requests.get('https://www.bbc.com/sport/football/premier-league/table')

In [ ]:
doc=lh.fromstring(resp.text)
listAll=[]
tr_elems=doc.xpath('//tr') # look for tr tags
for i in range(0,len(tr_elems)):
    dictI={} # create an empty dictionary
    for j  in range( 0 , len( tr_elems[i] ) ): # for each of the elements in the row j =0 to n-1
        dictI[j+1]= tr_elems[i][j].text_content() # Add the entry to key j 
    listAll.append( dictI  )
premTable= pd.DataFrame(listAll[1:-1])
premTable.columns=[str.lower() for str in listAll[0].values()] 
premTable

And formalizing to get other tables with the stub:

In [ ]:
def bbcGetTable(stub):
    resp=requests.get('https://www.bbc.com/sport/football/'+stub+'/table')
    doc=lh.fromstring(resp.text)
    listAll=[]
    tr_elems=doc.xpath('//tr') # look for tr tags
    for i in range(0,len(tr_elems)):
        dictI={} # create an empty dictionary
        for j  in range( 0 , len( tr_elems[i] ) ): # for each of the elements in the row j =0 to n-1
            dictI[j+1]= tr_elems[i][j].text_content() # Add the entry to key j 
        listAll.append( dictI  )
    outTable= pd.DataFrame(listAll[1:-1])
    outTable.columns=[str.lower() for str in listAll[0].values()] 
    return outTable

In [ ]:
bbcGetTable('german-bundesliga')

### Try it yourself

Write a regular expression to extract all email addresses from a sample string. Test it on a string containing at least three different email formats.

### Try it yourself

Write a regex that extracts all email addresses from a string.